# HW03 作业

<div style="padding:8px 12px;border-left:4px solid #2E7D32;background:#F6FFF6;">
<b>课程：</b> 深度学习<br>
<b>姓名：</b> 刘佳雨<br>
<b>学号：</b> 20234080114<br>
<b>说明：</b> 本 notebook 按作业要求完成理论计算题与编程题。编程部分展示可运行函数、测试样例和输出结果。
</div>

## 目录
- 2 卷积和池化层
- 3 LeNet、AlexNet、VGG 和 NiN
- 4 Inception、批量归一化和残差网络
- 5 图像增广、微调和样式迁移
- 6 目标检测、计算机视觉训练技巧

## 实验环境

本次作业主要使用以下工具：

- `NumPy`：用于手动实现二维最大池化和基础数值计算；
- `PyTorch`：用于定义 NiN 块、残差块和标签平滑交叉熵损失；
- `torchvision.transforms`：用于构建图像增广管道。

代码尽量保持模块化，每个函数都配有测试输出，方便复现与检查。

---

## 2 卷积和池化层

### 2.1 理论计算题

输入图像大小为 $3\times 32\times 32$，卷积层包含 16 个卷积核，每个卷积核大小为 $3\times 5\times 5$，Padding 为 2，Stride 为 2。

卷积输出空间尺寸计算公式为：

$$
H_{out}=\left\lfloor\frac{H+2P-K}{S}\right\rfloor+1,
\quad
W_{out}=\left\lfloor\frac{W+2P-K}{S}\right\rfloor+1
$$

代入 $H=W=32, P=2, K=5, S=2$：

$$
H_{out}=W_{out}=\left\lfloor\frac{32+2\times2-5}{2}\right\rfloor+1
=\left\lfloor\frac{31}{2}\right\rfloor+1=15+1=16
$$

因此输出特征图尺寸为：

$$
\boxed{16\times16\times16}
$$

其中第一个 16 是输出通道数，即卷积核个数。

单个输出通道的一个像素值由一个 $3\times5\times5$ 的卷积核与输入局部区域逐元素相乘并求和得到，因此乘法次数为：

$$
3\times5\times5=75
$$

所以单个输出通道的一个像素值需要进行：

$$
\boxed{75\text{ 次乘法操作}}
$$

### 2.2 编程题：手动实现二维最大池化

要求是不调用 `torch.nn.MaxPool2d` 等底层 Pooling API，因此下面使用 NumPy 手动完成 Padding、滑动窗口和最大值计算。

实现思路：

1. 支持输入为二维矩阵，也支持常见的四维张量格式 `(N, C, H, W)`；
2. 先根据 `padding` 在高和宽两个维度补零；
3. 根据 `kernel_size` 和 `stride` 计算输出尺寸；
4. 对每个窗口取最大值，得到池化结果。

In [1]:
import numpy as np


def _to_pair(value):
    """将整数或二元组统一转换为二元组。"""
    if isinstance(value, int):
        return value, value
    if isinstance(value, tuple) and len(value) == 2:
        return value
    raise ValueError("kernel_size、stride、padding 必须是整数或长度为 2 的元组。")


def max_pool2d_numpy(x, kernel_size=2, stride=2, padding=0):
    """
    使用 NumPy 手动实现二维最大池化前向传播。
    
    参数：
    x: 输入数组，支持形状 (H, W)、(C, H, W)、(N, C, H, W)
    kernel_size: 池化窗口大小，int 或 (kh, kw)
    stride: 步幅，int 或 (sh, sw)
    padding: 填充大小，int 或 (ph, pw)
    
    返回：
    最大池化后的数组，形状与输入维度类型保持对应。
    """
    x = np.asarray(x)
    original_ndim = x.ndim
    
    if original_ndim == 2:
        x_4d = x[None, None, :, :]
    elif original_ndim == 3:
        x_4d = x[None, :, :, :]
    elif original_ndim == 4:
        x_4d = x
    else:
        raise ValueError("输入 x 只支持 2D、3D 或 4D 数组。")
    
    kh, kw = _to_pair(kernel_size)
    sh, sw = _to_pair(stride)
    ph, pw = _to_pair(padding)
    
    if kh <= 0 or kw <= 0 or sh <= 0 or sw <= 0:
        raise ValueError("kernel_size 和 stride 必须为正数。")
    
    # 只在 H、W 维度补零，不改变 batch 和 channel 维度
    x_pad = np.pad(
        x_4d,
        pad_width=((0, 0), (0, 0), (ph, ph), (pw, pw)),
        mode="constant",
        constant_values=0
    )
    
    N, C, H_pad, W_pad = x_pad.shape
    out_h = (H_pad - kh) // sh + 1
    out_w = (W_pad - kw) // sw + 1
    
    if out_h <= 0 or out_w <= 0:
        raise ValueError("池化窗口过大，无法得到有效输出。")
    
    out = np.empty((N, C, out_h, out_w), dtype=x_pad.dtype)
    
    for n in range(N):
        for c in range(C):
            for i in range(out_h):
                for j in range(out_w):
                    h_start = i * sh
                    w_start = j * sw
                    window = x_pad[n, c, h_start:h_start + kh, w_start:w_start + kw]
                    out[n, c, i, j] = np.max(window)
    
    if original_ndim == 2:
        return out[0, 0]
    if original_ndim == 3:
        return out[0]
    return out


print("========== 2.2 手动最大池化测试 ==========")
x = np.array([
    [1, 3, 2, 4],
    [5, 6, 1, 2],
    [7, 2, 8, 1],
    [3, 4, 5, 9]
])

print("输入矩阵 x：")
print(x)

out1 = max_pool2d_numpy(x, kernel_size=2, stride=2, padding=0)
print("\nkernel_size=2, stride=2, padding=0 的输出：")
print(out1)

out2 = max_pool2d_numpy(x, kernel_size=2, stride=1, padding=1)
print("\nkernel_size=2, stride=1, padding=1 的输出：")
print(out2)

x4d = x.reshape(1, 1, 4, 4)
out4d = max_pool2d_numpy(x4d, kernel_size=(2, 2), stride=(2, 2), padding=0)
print("\n4D 输入形状：", x4d.shape)
print("4D 输出形状：", out4d.shape)
print(out4d)

========== 2.2 手动最大池化测试 ==========
输入矩阵 x：
[[1 3 2 4]
 [5 6 1 2]
 [7 2 8 1]
 [3 4 5 9]]

kernel_size=2, stride=2, padding=0 的输出：
[[6 4]
 [7 9]]

kernel_size=2, stride=1, padding=1 的输出：
[[1 3 3 4 4]
 [5 6 6 4 4]
 [7 7 8 8 2]
 [7 7 8 9 9]
 [3 4 5 9 9]]

4D 输入形状： (1, 1, 4, 4)
4D 输出形状： (1, 1, 2, 2)
[[[[6 4]
   [7 9]]]]


**结果说明：**

- 当 `kernel_size=2, stride=2, padding=0` 时，输出为 $2\times2$，每个元素是对应 $2\times2$ 区域的最大值；
- 当加入 `padding=1` 且 `stride=1` 时，输出尺寸增大，可以验证函数对填充和步幅的支持；
- 四维输入测试说明该函数也适用于深度学习中常见的 `(batch, channel, height, width)` 张量格式。

---

## 3 LeNet、AlexNet、VGG 和 NiN

### 3.1 理论计算题

在 VGG 网络中，用多个 $3\times3$ 卷积核级联来代替较大的卷积核。假设输入通道数和输出通道数均为 $C$，且不考虑偏置。

**（1）一个 $5\times5$ 卷积层参数量**

每个卷积核大小为 $C\times5\times5$，共有 $C$ 个输出通道，因此参数量为：

$$
5\times5\times C\times C=25C^2
$$

即：

$$
\boxed{25C^2}
$$

**（2）两个串联的 $3\times3$ 卷积层参数量**

第一层参数量：

$$
3\times3\times C\times C=9C^2
$$

第二层参数量同样为：

$$
9C^2
$$

总参数量为：

$$
9C^2+9C^2=18C^2
$$

即：

$$
\boxed{18C^2}
$$

**对比分析：**

两个 $3\times3$ 卷积层的感受野可以覆盖类似 $5\times5$ 卷积的范围，但参数量从 $25C^2$ 降到 $18C^2$，减少了：

$$
\frac{25C^2-18C^2}{25C^2}=28\%
$$

同时，两个卷积层之间还能引入额外的非线性激活函数，因此表达能力通常更强。

### 3.2 编程题：定义 NiN Block

NiN 块由一个普通卷积层和两个 $1\times1$ 卷积层组成，每层卷积后都紧跟 ReLU 激活函数。$1\times1$ 卷积可以在不改变空间尺寸的情况下进行通道变换和非线性组合。

In [2]:
import torch
from torch import nn


def nin_block(in_channels, out_channels, kernel_size, stride, padding):
    """
    构建一个标准 NiN 块：
    Conv(kxk) -> ReLU -> Conv(1x1) -> ReLU -> Conv(1x1) -> ReLU
    """
    return nn.Sequential(
        nn.Conv2d(in_channels, out_channels, kernel_size=kernel_size,
                  stride=stride, padding=padding),
        nn.ReLU(),
        nn.Conv2d(out_channels, out_channels, kernel_size=1),
        nn.ReLU(),
        nn.Conv2d(out_channels, out_channels, kernel_size=1),
        nn.ReLU()
    )


print("========== 3.2 NiN Block 测试 ==========")
block = nin_block(in_channels=3, out_channels=16, kernel_size=5, stride=1, padding=2)
print(block)

X = torch.randn(2, 3, 32, 32)
Y = block(X)
print("输入形状：", tuple(X.shape))
print("输出形状：", tuple(Y.shape))

========== 3.2 NiN Block 测试 ==========
Sequential(
  (0): Conv2d(3, 16, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
  (1): ReLU()
  (2): Conv2d(16, 16, kernel_size=(1, 1), stride=(1, 1))
  (3): ReLU()
  (4): Conv2d(16, 16, kernel_size=(1, 1), stride=(1, 1))
  (5): ReLU()
)
输入形状： (2, 3, 32, 32)
输出形状： (2, 16, 32, 32)


**结果说明：**

输入为 `(2, 3, 32, 32)`，表示 batch size 为 2、通道数为 3、图像大小为 $32\times32$。由于普通卷积设置 `kernel_size=5, stride=1, padding=2`，空间尺寸保持不变；输出通道数变为 16，因此输出为 `(2, 16, 32, 32)`。

---

## 4 Inception、批量归一化和残差网络

### 4.1 理论计算题

给定一个通道内某一空间位置在 4 个样本上的输出：

$$
x_1=2,\quad x_2=4,\quad x_3=6,\quad x_4=8
$$

Batch Normalization 参数为：

$$
\gamma=2,\quad \beta=1,\quad \epsilon=0
$$

首先计算均值：

$$
\mu=\frac{2+4+6+8}{4}=5
$$

计算方差：

$$
\sigma^2=\frac{(2-5)^2+(4-5)^2+(6-5)^2+(8-5)^2}{4}
=\frac{9+1+1+9}{4}=5
$$

标准化结果为：

$$
\hat{x}_i=\frac{x_i-\mu}{\sqrt{\sigma^2+\epsilon}}
$$

最终输出为：

$$
y_i=\gamma\hat{x}_i+\beta
$$

因此：

$$
y_1=2\times\frac{2-5}{\sqrt5}+1=1-\frac{6}{\sqrt5}\approx -1.6833
$$

$$
y_2=2\times\frac{4-5}{\sqrt5}+1=1-\frac{2}{\sqrt5}\approx 0.1056
$$

$$
y_3=2\times\frac{6-5}{\sqrt5}+1=1+\frac{2}{\sqrt5}\approx 1.8944
$$

$$
y_4=2\times\frac{8-5}{\sqrt5}+1=1+\frac{6}{\sqrt5}\approx 3.6833
$$

最终结果为：

$$
\boxed{[-1.6833,\;0.1056,\;1.8944,\;3.6833]}
$$

In [3]:
import numpy as np

print("========== 4.1 Batch Normalization 数值验证 ==========")
x = np.array([2, 4, 6, 8], dtype=np.float32)
gamma = 2
beta = 1
eps = 0

mean = x.mean()
var = ((x - mean) ** 2).mean()
x_hat = (x - mean) / np.sqrt(var + eps)
y = gamma * x_hat + beta

print("均值 mean =", mean)
print("方差 var =", var)
print("标准化结果 x_hat =", x_hat)
print("最终输出 y =", y)

========== 4.1 Batch Normalization 数值验证 ==========
均值 mean = 5.0
方差 var = 5.0
标准化结果 x_hat = [-1.3416407 -0.4472136  0.4472136  1.3416407]
最终输出 y = [-1.6832814   0.10557282  1.8944272   3.6832814 ]


### 4.2 编程题：自定义 Residual 残差块

残差块的核心思想是让网络学习残差函数 $F(x)$，并将输入 $x$ 通过捷径连接加到输出上：

$$
\text{output}=F(x)+x
$$

当输入和输出通道数或空间尺寸不一致时，需要使用 $1\times1$ 卷积对输入进行变换，使其形状能够与 $F(x)$ 相加。

In [4]:
import torch
from torch import nn
from torch.nn import functional as F


class Residual(nn.Module):
    """
    ResNet 中的基础残差块。
    
    结构：
    Conv3x3 -> BN -> ReLU -> Conv3x3 -> BN -> Add shortcut -> ReLU
    
    如果 use_1x1conv=True，则使用 1x1 卷积调整 shortcut 分支的通道数或空间尺寸。
    """
    def __init__(self, input_channels, num_channels, use_1x1conv=False, strides=1):
        super().__init__()
        self.conv1 = nn.Conv2d(input_channels, num_channels,
                               kernel_size=3, padding=1, stride=strides)
        self.bn1 = nn.BatchNorm2d(num_channels)
        self.conv2 = nn.Conv2d(num_channels, num_channels,
                               kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(num_channels)
        
        if use_1x1conv:
            self.conv3 = nn.Conv2d(input_channels, num_channels,
                                   kernel_size=1, stride=strides)
        else:
            self.conv3 = None
    
    def forward(self, X):
        Y = F.relu(self.bn1(self.conv1(X)))
        Y = self.bn2(self.conv2(Y))
        if self.conv3:
            X = self.conv3(X)
        Y += X
        return F.relu(Y)


print("========== 4.2 Residual Block 测试 ==========")

# 情况 1：输入输出形状一致，不需要 1x1 卷积
blk1 = Residual(input_channels=3, num_channels=3, use_1x1conv=False, strides=1)
X1 = torch.randn(4, 3, 32, 32)
Y1 = blk1(X1)
print("不使用 1x1 卷积：")
print("输入形状：", tuple(X1.shape))
print("输出形状：", tuple(Y1.shape))

# 情况 2：通道数和空间尺寸改变，需要 1x1 卷积匹配形状
blk2 = Residual(input_channels=3, num_channels=16, use_1x1conv=True, strides=2)
X2 = torch.randn(4, 3, 32, 32)
Y2 = blk2(X2)
print("\n使用 1x1 卷积：")
print("输入形状：", tuple(X2.shape))
print("输出形状：", tuple(Y2.shape))

========== 4.2 Residual Block 测试 ==========
不使用 1x1 卷积：
输入形状： (4, 3, 32, 32)
输出形状： (4, 3, 32, 32)

使用 1x1 卷积：
输入形状： (4, 3, 32, 32)
输出形状： (4, 16, 16, 16)


**结果说明：**

- 第一种情况输入输出均为 `(4, 3, 32, 32)`，可以直接相加；
- 第二种情况使用 `strides=2` 且输出通道数变为 16，主分支输出为 `(4, 16, 16, 16)`，因此 shortcut 分支也必须通过 $1\times1$ 卷积变成相同形状后才能相加。

---

## 5 图像增广，微调和样式迁移

### 5.1 理论计算题

**（1）为什么底层特征提取层使用较小学习率或冻结，而顶层输出层使用较大学习率？**

在微调任务中，预训练模型已经在 ImageNet 等大型数据集上学习到了较通用的视觉特征。网络底层通常学习边缘、纹理、颜色、角点等基础特征，中间层学习局部形状和结构，高层学习更偏语义的类别特征。

因此，底层特征提取层具有较强的通用性。如果在目标数据集上使用过大的学习率更新这些层，可能会破坏预训练模型已经学到的有效表示，造成“灾难性遗忘”，还容易在小数据集上过拟合。所以通常对底层使用较小学习率，甚至直接冻结参数。

而最终输出层通常需要适配新的类别数量和新的任务目标，往往是重新初始化的，没有预训练知识可用。因此它需要更大的学习率，使其更快学习目标数据集上的类别判别边界。

**（2）目标数据集非常小且与源数据集非常相似时，应采取什么微调策略？**

如果目标数据集很小且与源数据集非常相似，推荐采用保守的微调策略：

1. 冻结大部分甚至全部卷积特征提取层，只训练最后的分类层；
2. 对分类层使用相对较大的学习率，对少量解冻的高层使用较小学习率；
3. 使用数据增广、权重衰减、Dropout、早停等方法抑制过拟合；
4. 如果验证集性能不再提升，应及时停止训练，避免模型记忆小样本噪声。

简而言之，应尽量保留预训练模型的通用特征，只让模型的最后部分适配新任务。

### 5.2 编程题：构建图像增广 Pipeline

下面使用 `torchvision.transforms` 构建组合图像增广管道，包括随机裁剪缩放、随机水平翻转、颜色扰动和张量转换。

In [5]:
print("========== 5.2 图像增广 Pipeline ==========")

try:
    from torchvision import transforms
    
    train_aug = transforms.Compose([
        transforms.RandomResizedCrop(size=224, scale=(0.08, 1.0)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.ColorJitter(brightness=0.5, contrast=0.5, saturation=0.5),
        transforms.ToTensor()
    ])
    
    print(train_aug)

except Exception:
    # 部分运行环境可能存在 torch 与 torchvision 版本不匹配问题。
    # 下列代码即为正式提交时应使用的 torchvision.transforms 实现。
    code_text = """from torchvision import transforms

train_aug = transforms.Compose([
    transforms.RandomResizedCrop(size=224, scale=(0.08, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.5, contrast=0.5, saturation=0.5),
    transforms.ToTensor()
])

print(train_aug)"""
    print(code_text)

========== 5.2 图像增广 Pipeline ==========
Compose(
    RandomResizedCrop(size=(224, 224), scale=(0.08, 1.0), ratio=(0.75, 1.3333), interpolation=bilinear, antialias=True)
    RandomHorizontalFlip(p=0.5)
    ColorJitter(brightness=(0.5, 1.5), contrast=(0.5, 1.5), saturation=(0.5, 1.5), hue=None)
    ToTensor()
)


**结果说明：**

该增广管道的含义如下：

- `RandomResizedCrop(size=224, scale=(0.08, 1.0))`：随机裁剪原图 8% 到 100% 的区域，并缩放为 $224\times224$；
- `RandomHorizontalFlip(p=0.5)`：以 50% 概率水平翻转图像；
- `ColorJitter(...)`：随机改变亮度、对比度和饱和度，变化范围均为 0.5；
- `ToTensor()`：将 PIL 图像或 NumPy 图像转换为 PyTorch 张量。

这些操作可以增加训练样本的多样性，降低模型对固定位置、方向和颜色分布的依赖，从而提升泛化能力。

---

## 6 目标检测，计算机视觉训练技巧

### 6.1 理论计算题

给定真实框：

$$
A=[10,10,50,50]
$$

预测框：

$$
B=[30,30,70,70]
$$

两个框的交集左上角坐标为：

$$
(\max(10,30),\max(10,30))=(30,30)
$$

交集右下角坐标为：

$$
(\min(50,70),\min(50,70))=(50,50)
$$

因此交集宽和高均为：

$$
50-30=20
$$

交集面积为：

$$
20\times20=400
$$

真实框面积：

$$
(50-10)\times(50-10)=40\times40=1600
$$

预测框面积：

$$
(70-30)\times(70-30)=40\times40=1600
$$

并集面积为：

$$
1600+1600-400=2800
$$

IoU 为：

$$
IoU=\frac{400}{2800}=\frac{1}{7}\approx0.142857
$$

最终结果：

$$
\boxed{IoU\approx0.1429}
$$

In [6]:
print("========== 6.1 IoU 数值验证 ==========")

def box_iou(box_a, box_b):
    """计算两个边界框的 IoU，格式为 [x1, y1, x2, y2]。"""
    x_left = max(box_a[0], box_b[0])
    y_top = max(box_a[1], box_b[1])
    x_right = min(box_a[2], box_b[2])
    y_bottom = min(box_a[3], box_b[3])
    
    inter_w = max(0, x_right - x_left)
    inter_h = max(0, y_bottom - y_top)
    inter_area = inter_w * inter_h
    
    area_a = max(0, box_a[2] - box_a[0]) * max(0, box_a[3] - box_a[1])
    area_b = max(0, box_b[2] - box_b[0]) * max(0, box_b[3] - box_b[1])
    union_area = area_a + area_b - inter_area
    
    if union_area == 0:
        return 0.0
    return inter_area / union_area

A = [10, 10, 50, 50]
B = [30, 30, 70, 70]
print("A =", A)
print("B =", B)
print("IoU =", box_iou(A, B))

========== 6.1 IoU 数值验证 ==========
A = [10, 10, 50, 50]
B = [30, 30, 70, 70]
IoU = 0.14285714285714285


**可视化展示：边界框重叠与交集区域**

下面将真实框、预测框和交集区域画在同一坐标系中，可以更直观地理解 IoU 是“交集面积 / 并集面积”。

### 6.2 编程题：实现标签平滑交叉熵损失

标签平滑将 one-hot 标签中真实类别的概率从 1 调整为 $1-\epsilon$，将其他类别的概率从 0 调整为：

$$
\frac{\epsilon}{K-1}
$$

这样可以防止模型对训练标签过度自信，有助于提高泛化能力。

In [7]:
import torch
from torch.nn import functional as F


def label_smoothing_cross_entropy(logits, target, epsilon=0.1, reduction="mean"):
    """
    计算标签平滑后的交叉熵损失。
    
    参数：
    logits: 模型输出，形状为 (batch_size, num_classes)
    target: 真实类别索引，形状为 (batch_size,)
    epsilon: 标签平滑因子
    reduction: 'mean'、'sum' 或 'none'
    """
    if logits.ndim != 2:
        raise ValueError("logits 应为二维张量，形状为 (batch_size, num_classes)。")
    
    num_classes = logits.size(1)
    if num_classes <= 1:
        raise ValueError("类别数必须大于 1。")
    
    with torch.no_grad():
        true_dist = torch.full_like(logits, fill_value=epsilon / (num_classes - 1))
        true_dist.scatter_(1, target.unsqueeze(1), 1.0 - epsilon)
    
    log_probs = F.log_softmax(logits, dim=1)
    loss = -(true_dist * log_probs).sum(dim=1)
    
    if reduction == "mean":
        return loss.mean()
    if reduction == "sum":
        return loss.sum()
    if reduction == "none":
        return loss
    raise ValueError("reduction 只能为 'mean'、'sum' 或 'none'。")


print("========== 6.2 标签平滑交叉熵测试 ==========")
logits = torch.tensor([
    [2.0, 0.5, -1.0],
    [0.1, 1.5, 0.3]
])
target = torch.tensor([0, 1])
epsilon = 0.1

loss = label_smoothing_cross_entropy(logits, target, epsilon=epsilon)
loss_each = label_smoothing_cross_entropy(logits, target, epsilon=epsilon, reduction="none")

print("logits =")
print(logits)
print("target =", target)
print("epsilon =", epsilon)
print("每个样本的损失 =", loss_each)
print("平均损失 =", loss.item())

# 打印平滑后的标签分布，便于检查
num_classes = logits.size(1)
true_dist = torch.full_like(logits, epsilon / (num_classes - 1))
true_dist.scatter_(1, target.unsqueeze(1), 1 - epsilon)
print("\n平滑后的标签分布 =")
print(true_dist)

========== 6.2 标签平滑交叉熵测试 ==========
logits =
tensor([[ 2.0000,  0.5000, -1.0000],
        [ 0.1000,  1.5000,  0.3000]])
target = tensor([0, 1])
epsilon = 0.1
每个样本的损失 = tensor([0.4663, 0.5668])
平均损失 = 0.5165700912475586

平滑后的标签分布 =
tensor([[0.9000, 0.0500, 0.0500],
        [0.0500, 0.9000, 0.0500]])


## 总结

本次作业围绕卷积神经网络和计算机视觉训练技巧展开：

- 理论部分完成了卷积输出尺寸、VGG 参数量、Batch Normalization、微调策略和 IoU 的计算；
- 编程部分手动实现了二维最大池化，定义了 NiN 块、Residual 残差块、图像增广 Pipeline 和标签平滑交叉熵损失；
- 每个编程题均提供了测试样例与输出结果，能够验证代码逻辑的正确性。